In [1]:
print("Working Fine")

Working Fine


In [3]:
import sys
print(sys.executable)

/Users/vinayak/Desktop/Major Project/venv/bin/python


In [5]:
import pandas as pd

In [6]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

print("Movies Data:")
print(movies.head())

print("\nRatings Data:")
print(ratings.head())

print("\nMovies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)


Movies Data:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Ratings Data:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

Movies Shape: (9742, 3)
Ratings Shape: (100836, 4)


In [7]:
print("Minimun rating:", ratings['rating'].min())
print("Maximum rating:", ratings['rating'].max())

Minimun rating: 0.5
Maximum rating: 5.0


In [8]:
print("Total unique users:", ratings['userId'].nunique())
print("Total unique movies rated:", ratings['movieId'].nunique())


Total unique users: 610
Total unique movies rated: 9724


In [9]:
movie_data = pd.merge(ratings, movies, on='movieId')

print(movie_data.head())
print("\nShape after merging:", movie_data.shape)


   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  

Shape after merging: (100836, 6)


In [10]:
print("\nShape after merging:", movie_data.shape)



Shape after merging: (100836, 6)


In [11]:
user_movie_matrix = movie_data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

print(user_movie_matrix.shape)


(610, 9719)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill missing ratings with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Calculate user similarity
user_similarity = cosine_similarity(user_movie_matrix_filled)

print("User similarity shape:", user_similarity.shape)


User similarity shape: (610, 610)


In [14]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix_filled.index,
    columns=user_movie_matrix_filled.index
)

print(user_similarity_df.head())


userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000  0.027283  0.059720  0.194395  0.129080  0.128152  0.158744   
2       0.027283  1.000000  0.000000  0.003726  0.016614  0.025333  0.027585   
3       0.059720  0.000000  1.000000  0.002251  0.005020  0.003936  0.000000   
4       0.194395  0.003726  0.002251  1.000000  0.128659  0.088491  0.115120   
5       0.129080  0.016614  0.005020  0.128659  1.000000  0.300349  0.108342   

userId       8         9         10   ...       601       602       603  \
userId                                ...                                 
1       0.136968  0.064263  0.016875  ...  0.080554  0.164455  0.221486   
2       0.027257  0.000000  0.067445  ...  0.202671  0.016866  0.011997   
3       0.004941  0.000000  0.000000  ...  0.005048  0.004892  0.024992   
4       0.062969  0.011361  0.031163  ...  0.085938  0.128273  0

In [15]:
similar_users = user_similarity_df[1].sort_values(ascending=False)

print(similar_users.head(10))



userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
Name: 1, dtype: float64


In [16]:
similar_users = user_similarity_df[1].sort_values(ascending=False)

# remove self similarity
similar_users = similar_users.iloc[1:6]

print(similar_users)


userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64


In [17]:
def recommend_for_user(user_id, num_recommendations=5):

    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).iloc[1:6]

    # Get ratings from similar users
    similar_users_ratings = user_movie_matrix.loc[similar_users.index]

    # Average their ratings
    recommendation_scores = similar_users_ratings.mean().sort_values(ascending=False)

    # Remove movies already rated by user
    user_rated = user_movie_matrix.loc[user_id]
    recommendation_scores = recommendation_scores[user_rated.isna()]

    return recommendation_scores.head(num_recommendations)


In [18]:
recommend_for_user(1)


title
Ref, The (1994)                                                  5.0
Pirates of the Caribbean: The Curse of the Black Pearl (2003)    5.0
Dirty Work (1998)                                                5.0
Wallace & Gromit: The Best of Aardman Animation (1996)           5.0
Doctor Zhivago (1965)                                            5.0
dtype: float64

In [19]:
def recommend_for_user(user_id, num_recommendations=5):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False).iloc[1:6]

    weighted_ratings = pd.Series(dtype=float)

    for similar_user, similarity_score in similar_users.items():
        weighted_ratings = weighted_ratings.add(
            user_movie_matrix.loc[similar_user] * similarity_score,
            fill_value=0
        )

    user_rated = user_movie_matrix.loc[user_id]
    recommendations = weighted_ratings[user_rated.isna()]

    return recommendations.sort_values(ascending=False).head(num_recommendations)


In [20]:
recommend_for_user(1)

title
Aliens (1986)                        8.324253
Hunt for Red October, The (1990)     7.453767
Blade Runner (1982)                  6.943652
Terminator 2: Judgment Day (1991)    6.941864
Die Hard (1988)                      6.935430
dtype: float64